<a href="https://colab.research.google.com/github/Ysagar-hub/Crop-Plantation-Stage/blob/main/Field_Boundaries_of_all_5_of_Satsure_and_Our_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
LEAN target-metrics pipeline -- Field Boundaries + Area Delineated (ha) only
--------------------------------------------------------------------------------
Unlike the full pipeline, this does NOT write per-village shapefiles. It only
needs two numbers per district:
    - Field Boundaries Identified  = total plot count
    - Area Delineated (ha)         = total area, in HECTARES

Much lighter/faster since nothing is written to disk except the raw API
responses (for safety) and the final summary -- no shapefile writing, no
per-village GeoDataFrame overhead.

USAGE -- edit these two lines, run:
    DISTRICT_NAME = "Gaya"
    VILLAGE_CSV = "village_codes_Gaya.csv"

Appends its result to district_area_summary.csv each time you run it for a
new district, so after running all 4 you have one combined comparison table.
"""

DISTRICT_NAME = "Gaya"
VILLAGE_CSV = "/content/village_codes_Gaya (1).csv"


# ============================================================
# INSTALL + IMPORTS
# ============================================================
import subprocess
subprocess.run(["pip", "install", "-q", "requests", "shapely", "pyproj", "pandas"], check=True)

import requests, json, csv, os, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely import wkt as shapely_wkt
from shapely.geometry import shape
import pandas as pd

RESULTS_JSONL = f"results_{DISTRICT_NAME.lower()}.jsonl"
STATE_CODE = "10"
MAX_WORKERS = 5
TIMEOUT_SECONDS = 60
MAX_RETRY_PASSES = 3
SQM_PER_HECTARE = 10000.0
SQM_PER_ACRE = 4046.8564224
SUMMARY_CSV = "district_area_summary.csv"

print(f"{'='*70}\nRUNNING DISTRICT: {DISTRICT_NAME}\n{'='*70}")


# ============================================================
# TOKEN-MANAGED API CLIENT
# ============================================================
BASE_URL = "https://bhunaksha.bihar.gov.in/rest/GetMapInstanceMap"
GENERATE_TOKEN_URL = f"{BASE_URL}/generateToken"
GET_GEOM_URL = f"{BASE_URL}/getGeom"

class BhunakshaClient:
    def __init__(self):
        self.token, self.token_expiry = None, None
        self._lock = threading.Lock()

    def _parse_expiry(self, s, at):
        try:
            v, unit = s.strip().split()
            v = int(v)
            unit = unit.lower()
            delta = timedelta(minutes=v) if "minute" in unit else timedelta(seconds=v) if "second" in unit else timedelta(hours=v) if "hour" in unit else timedelta(minutes=30)
        except Exception:
            delta = timedelta(minutes=30)
        return at + delta - timedelta(seconds=60)

    def _generate_token_locked(self):
        at = datetime.now()
        r = requests.post(GENERATE_TOKEN_URL, timeout=15)
        r.raise_for_status()
        data = r.json()
        if not data.get("success"):
            raise RuntimeError(f"Token generation failed: {data}")
        self.token = data["token"]
        self.token_expiry = self._parse_expiry(data.get("expiresIn", "30 minutes"), at)

    def _get_token(self):
        with self._lock:
            if self.token is None or datetime.now() >= self.token_expiry:
                self._generate_token_locked()
            return self.token

    def get_geom(self, state, code, map_type, sheet_no):
        token = self._get_token()
        payload = {"state": state, "lgdVillageCode": code, "mapType": map_type, "sheetNo": sheet_no, "token": token}
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        if r.status_code in (401, 403):
            with self._lock:
                self._generate_token_locked()
                token = self.token
            payload["token"] = token
            r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        r.raise_for_status()
        return r.json()


# ============================================================
# STEP 1: FETCH (raw responses saved for safety, multi-pass retry)
# ============================================================
def load_village_rows(csv_path):
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        raw_rows = list(csv.DictReader(f))
    def norm(k): return (k or "").strip().lower().replace(" ", "_")
    headers = list(raw_rows[0].keys())
    nmap = {norm(k): k for k in headers}
    if "lgd_village_code" not in nmap:
        raise KeyError(f"No lgd_village_code column found. Actual columns: {headers}")
    rows = []
    for row in raw_rows:
        rows.append({
            "lgd_village_code": str(row[nmap["lgd_village_code"]]).strip(),
            "map_type": row.get(nmap.get("map_type", ""), "CS").strip() if nmap.get("map_type") else "CS",
            "sheet_no": row.get(nmap.get("sheet_no", ""), "00").strip() if nmap.get("sheet_no") else "00",
        })
    return rows

def get_fetched_codes():
    done = set()
    if os.path.exists(RESULTS_JSONL):
        with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["lgd_village_code"])
                except Exception:
                    continue
    return done

village_rows = load_village_rows(VILLAGE_CSV)
print(f"Loaded {len(village_rows)} villages from {VILLAGE_CSV}")

client = BhunakshaClient()
write_lock = threading.Lock()

def fetch_one(row):
    return row["lgd_village_code"], row["map_type"], row["sheet_no"], client.get_geom(STATE_CODE, row["lgd_village_code"], row["map_type"], row["sheet_no"])

for attempt in range(1, MAX_RETRY_PASSES + 1):
    already_done = get_fetched_codes()
    todo = [r for r in village_rows if r["lgd_village_code"] not in already_done]
    if not todo:
        print("All villages fetched.")
        break

    print(f"\n--- Fetch pass {attempt}/{MAX_RETRY_PASSES}: {len(todo)} villages remaining ---")
    done_count, fail_count = 0, 0

    with open(RESULTS_JSONL, "a", encoding="utf-8") as out_f:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_one, row): row for row in todo}
            for future in as_completed(futures):
                row = futures[future]
                try:
                    code, mt, sn, geom = future.result()
                    with write_lock:
                        out_f.write(json.dumps({"lgd_village_code": code, "map_type": mt, "sheet_no": sn, "geometry": geom}) + "\n")
                        out_f.flush()
                        done_count += 1
                        if done_count % 50 == 0:
                            print(f"  [{done_count}/{len(todo)}] fetched this pass")
                except Exception:
                    fail_count += 1

    print(f"Pass {attempt} done: {done_count} succeeded, {fail_count} failed.")

final_fetched = get_fetched_codes()
final_missing = len(village_rows) - len(final_fetched)
print(f"\nFETCH SUMMARY: {len(final_fetched)}/{len(village_rows)} fetched ({final_missing} missing)")


# ============================================================
# STEP 2: COMPUTE TOTALS ONLY -- no shapefile writing
# ============================================================
print(f"\n--- Computing field boundary count + area ---")

total_plots = 0
total_area_sqm = 0.0
villages_processed = 0

with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        features = (record.get("geometry") or {}).get("features", [])
        if not features:
            continue
        villages_processed += 1

        village_areas = []
        for feat in features:
            props = feat.get("properties", {})
            geom = None
            wkt_str = props.get("the_geom")
            if wkt_str:
                try: geom = shapely_wkt.loads(wkt_str)
                except Exception: geom = None
            if geom is None:
                raw_geom = feat.get("geometry")
                if raw_geom:
                    try: geom = shape(raw_geom)
                    except Exception: geom = None
            if geom is None:
                continue

            total_plots += 1
            raw_area = props.get("area_acres") if props.get("area_acres") is not None else props.get("area")
            if raw_area is not None:
                try:
                    village_areas.append(float(raw_area))
                except (ValueError, TypeError):
                    pass

        if village_areas:
            median_val = sorted(village_areas)[len(village_areas)//2]
            # AUTO UNIT DETECTION: same heuristic as before -- values in the
            # thousands+ are sq meters, smaller values are already acres
            if median_val > 50:
                village_sum_sqm = sum(village_areas)  # already sqm
            else:
                village_sum_sqm = sum(village_areas) * SQM_PER_ACRE  # acres -> sqm
            total_area_sqm += village_sum_sqm

total_area_ha = total_area_sqm / SQM_PER_HECTARE

print(f"\n{'='*60}")
print(f"District: {DISTRICT_NAME}")
print(f"Villages processed: {villages_processed}/{len(village_rows)}")
print(f"Field Boundaries Identified (total plots): {total_plots:,}")
print(f"Area Delineated (ha): {total_area_ha:,.2f}")
print(f"{'='*60}")

# append to running summary CSV
row = {
    "District": DISTRICT_NAME,
    "Field Boundaries Identified": total_plots,
    "Area Delineated (ha)": round(total_area_ha, 2),
    "Villages Processed": villages_processed,
    "Villages Total": len(village_rows),
}
if os.path.exists(SUMMARY_CSV):
    df = pd.read_csv(SUMMARY_CSV)
    df = df[df["District"] != DISTRICT_NAME]  # replace if re-running same district
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])
df.to_csv(SUMMARY_CSV, index=False)
print(f"\nAppended to {SUMMARY_CSV} -- run this same script for each remaining district.")

try:
    from google.colab import files
    files.download(SUMMARY_CSV)
    files.download(RESULTS_JSONL)
except Exception:
    pass


RUNNING DISTRICT: Gaya
Loaded 2324 villages from /content/village_codes_Gaya (1).csv

--- Fetch pass 1/3: 2324 villages remaining ---
  [50/2324] fetched this pass
  [100/2324] fetched this pass
  [150/2324] fetched this pass
  [200/2324] fetched this pass
  [250/2324] fetched this pass
  [300/2324] fetched this pass
  [350/2324] fetched this pass
  [400/2324] fetched this pass
  [450/2324] fetched this pass
  [500/2324] fetched this pass
  [550/2324] fetched this pass
  [600/2324] fetched this pass
  [650/2324] fetched this pass
  [700/2324] fetched this pass
  [750/2324] fetched this pass
  [800/2324] fetched this pass
  [850/2324] fetched this pass
  [900/2324] fetched this pass
  [950/2324] fetched this pass
  [1000/2324] fetched this pass
  [1050/2324] fetched this pass
  [1100/2324] fetched this pass
  [1150/2324] fetched this pass
  [1200/2324] fetched this pass
  [1250/2324] fetched this pass
  [1300/2324] fetched this pass
  [1350/2324] fetched this pass
  [1400/2324] fetched 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import os

MASTER_FILE = "/content/DCS_BhuNaksha_All_050826.xlsx"

# Rules based on how the pivot tables are structured in each sheet
extraction_rules = {
    'Madhubani': {'RS': 'Row Labels'},
    'Saran': {'CS': 'Row Labels', 'RS': 'Row Labels.1'},
    'Samastipur': {'CS': 'Unnamed: 0', 'RS': 'Row Labels'}
}

print(f"Reading {MASTER_FILE}...\n")
xls = pd.ExcelFile(MASTER_FILE)

for district, rules in extraction_rules.items():
    print(f"--- Extracting {district} ---")
    df = pd.read_excel(xls, sheet_name=district)

    extracted_data = []
    for map_type, col_name in rules.items():
        if col_name in df.columns:
            # Convert to numeric, dropping text and blanks
            codes = pd.to_numeric(df[col_name], errors='coerce').dropna()

            # Keep only valid 6-digit Bihar LGD codes (filters out Grand Totals)
            valid_codes = codes[(codes >= 100000) & (codes <= 999999)].astype(int).astype(str)

            for code in valid_codes:
                extracted_data.append({
                    "lgd_village_code": code,
                    "map_type": map_type,
                    "sheet_no": "00"
                })

    if extracted_data:
        out_df = pd.DataFrame(extracted_data)
        out_file = f"village_codes_{district}.csv"
        out_df.to_csv(out_file, index=False)
        print(f"✅ Saved {len(out_df)} village codes to {out_file}\n")
    else:
        print(f"⚠️ Could not find valid codes for {district}\n")

Reading /content/DCS_BhuNaksha_All_050826.xlsx...

--- Extracting Madhubani ---
✅ Saved 1102 village codes to village_codes_Madhubani.csv

--- Extracting Saran ---
✅ Saved 3378 village codes to village_codes_Saran.csv

--- Extracting Samastipur ---
✅ Saved 1837 village codes to village_codes_Samastipur.csv



In [ ]:
"""
LEAN target-metrics pipeline -- Field Boundaries + Area Delineated (ha) only
--------------------------------------------------------------------------------
Unlike the full pipeline, this does NOT write per-village shapefiles. It only
needs two numbers per district:
    - Field Boundaries Identified  = total plot count
    - Area Delineated (ha)         = total area, in HECTARES

Much lighter/faster since nothing is written to disk except the raw API
responses (for safety) and the final summary -- no shapefile writing, no
per-village GeoDataFrame overhead.

USAGE -- edit these two lines, run:
    DISTRICT_NAME = "Gaya"
    VILLAGE_CSV = "village_codes_Gaya.csv"

Appends its result to district_area_summary.csv each time you run it for a
new district, so after running all 4 you have one combined comparison table.
"""

DISTRICT_NAME = "Madhubani"
VILLAGE_CSV = "/content/village_codes_Madhubani.csv"


# ============================================================
# INSTALL + IMPORTS
# ============================================================
import subprocess
subprocess.run(["pip", "install", "-q", "requests", "shapely", "pyproj", "pandas"], check=True)

import requests, json, csv, os, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely import wkt as shapely_wkt
from shapely.geometry import shape
import pandas as pd

RESULTS_JSONL = f"results_{DISTRICT_NAME.lower()}.jsonl"
STATE_CODE = "10"
MAX_WORKERS = 5
TIMEOUT_SECONDS = 60
MAX_RETRY_PASSES = 3
SQM_PER_HECTARE = 10000.0
SQM_PER_ACRE = 4046.8564224
SUMMARY_CSV = "district_area_summary.csv"

print(f"{'='*70}\nRUNNING DISTRICT: {DISTRICT_NAME}\n{'='*70}")


# ============================================================
# TOKEN-MANAGED API CLIENT
# ============================================================
BASE_URL = "https://bhunaksha.bihar.gov.in/rest/GetMapInstanceMap"
GENERATE_TOKEN_URL = f"{BASE_URL}/generateToken"
GET_GEOM_URL = f"{BASE_URL}/getGeom"

class BhunakshaClient:
    def __init__(self):
        self.token, self.token_expiry = None, None
        self._lock = threading.Lock()

    def _parse_expiry(self, s, at):
        try:
            v, unit = s.strip().split()
            v = int(v)
            unit = unit.lower()
            delta = timedelta(minutes=v) if "minute" in unit else timedelta(seconds=v) if "second" in unit else timedelta(hours=v) if "hour" in unit else timedelta(minutes=30)
        except Exception:
            delta = timedelta(minutes=30)
        return at + delta - timedelta(seconds=60)

    def _generate_token_locked(self):
        at = datetime.now()
        r = requests.post(GENERATE_TOKEN_URL, timeout=15)
        r.raise_for_status()
        data = r.json()
        if not data.get("success"):
            raise RuntimeError(f"Token generation failed: {data}")
        self.token = data["token"]
        self.token_expiry = self._parse_expiry(data.get("expiresIn", "30 minutes"), at)

    def _get_token(self):
        with self._lock:
            if self.token is None or datetime.now() >= self.token_expiry:
                self._generate_token_locked()
            return self.token

    def get_geom(self, state, code, map_type, sheet_no):
        token = self._get_token()
        payload = {"state": state, "lgdVillageCode": code, "mapType": map_type, "sheetNo": sheet_no, "token": token}
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        if r.status_code in (401, 403):
            with self._lock:
                self._generate_token_locked()
                token = self.token
            payload["token"] = token
            r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        r.raise_for_status()
        return r.json()


# ============================================================
# STEP 1: FETCH (raw responses saved for safety, multi-pass retry)
# ============================================================
def load_village_rows(csv_path):
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        raw_rows = list(csv.DictReader(f))
    def norm(k): return (k or "").strip().lower().replace(" ", "_")
    headers = list(raw_rows[0].keys())
    nmap = {norm(k): k for k in headers}
    if "lgd_village_code" not in nmap:
        raise KeyError(f"No lgd_village_code column found. Actual columns: {headers}")
    rows = []
    for row in raw_rows:
        rows.append({
            "lgd_village_code": str(row[nmap["lgd_village_code"]]).strip(),
            "map_type": row.get(nmap.get("map_type", ""), "CS").strip() if nmap.get("map_type") else "CS",
            "sheet_no": row.get(nmap.get("sheet_no", ""), "00").strip() if nmap.get("sheet_no") else "00",
        })
    return rows

def get_fetched_codes():
    done = set()
    if os.path.exists(RESULTS_JSONL):
        with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["lgd_village_code"])
                except Exception:
                    continue
    return done

village_rows = load_village_rows(VILLAGE_CSV)
print(f"Loaded {len(village_rows)} villages from {VILLAGE_CSV}")

client = BhunakshaClient()
write_lock = threading.Lock()

def fetch_one(row):
    return row["lgd_village_code"], row["map_type"], row["sheet_no"], client.get_geom(STATE_CODE, row["lgd_village_code"], row["map_type"], row["sheet_no"])

for attempt in range(1, MAX_RETRY_PASSES + 1):
    already_done = get_fetched_codes()
    todo = [r for r in village_rows if r["lgd_village_code"] not in already_done]
    if not todo:
        print("All villages fetched.")
        break

    print(f"\n--- Fetch pass {attempt}/{MAX_RETRY_PASSES}: {len(todo)} villages remaining ---")
    done_count, fail_count = 0, 0

    with open(RESULTS_JSONL, "a", encoding="utf-8") as out_f:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_one, row): row for row in todo}
            for future in as_completed(futures):
                row = futures[future]
                try:
                    code, mt, sn, geom = future.result()
                    with write_lock:
                        out_f.write(json.dumps({"lgd_village_code": code, "map_type": mt, "sheet_no": sn, "geometry": geom}) + "\n")
                        out_f.flush()
                        done_count += 1
                        if done_count % 50 == 0:
                            print(f"  [{done_count}/{len(todo)}] fetched this pass")
                except Exception:
                    fail_count += 1

    print(f"Pass {attempt} done: {done_count} succeeded, {fail_count} failed.")

final_fetched = get_fetched_codes()
final_missing = len(village_rows) - len(final_fetched)
print(f"\nFETCH SUMMARY: {len(final_fetched)}/{len(village_rows)} fetched ({final_missing} missing)")


# ============================================================
# STEP 2: COMPUTE TOTALS ONLY -- no shapefile writing
# ============================================================
print(f"\n--- Computing field boundary count + area ---")

total_plots = 0
total_area_sqm = 0.0
villages_processed = 0

with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        features = (record.get("geometry") or {}).get("features", [])
        if not features:
            continue
        villages_processed += 1

        village_areas = []
        for feat in features:
            props = feat.get("properties", {})
            geom = None
            wkt_str = props.get("the_geom")
            if wkt_str:
                try: geom = shapely_wkt.loads(wkt_str)
                except Exception: geom = None
            if geom is None:
                raw_geom = feat.get("geometry")
                if raw_geom:
                    try: geom = shape(raw_geom)
                    except Exception: geom = None
            if geom is None:
                continue

            total_plots += 1
            raw_area = props.get("area_acres") if props.get("area_acres") is not None else props.get("area")
            if raw_area is not None:
                try:
                    village_areas.append(float(raw_area))
                except (ValueError, TypeError):
                    pass

        if village_areas:
            median_val = sorted(village_areas)[len(village_areas)//2]
            # AUTO UNIT DETECTION: same heuristic as before -- values in the
            # thousands+ are sq meters, smaller values are already acres
            if median_val > 50:
                village_sum_sqm = sum(village_areas)  # already sqm
            else:
                village_sum_sqm = sum(village_areas) * SQM_PER_ACRE  # acres -> sqm
            total_area_sqm += village_sum_sqm

total_area_ha = total_area_sqm / SQM_PER_HECTARE

print(f"\n{'='*60}")
print(f"District: {DISTRICT_NAME}")
print(f"Villages processed: {villages_processed}/{len(village_rows)}")
print(f"Field Boundaries Identified (total plots): {total_plots:,}")
print(f"Area Delineated (ha): {total_area_ha:,.2f}")
print(f"{'='*60}")

# append to running summary CSV
row = {
    "District": DISTRICT_NAME,
    "Field Boundaries Identified": total_plots,
    "Area Delineated (ha)": round(total_area_ha, 2),
    "Villages Processed": villages_processed,
    "Villages Total": len(village_rows),
}
if os.path.exists(SUMMARY_CSV):
    df = pd.read_csv(SUMMARY_CSV)
    df = df[df["District"] != DISTRICT_NAME]  # replace if re-running same district
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])
df.to_csv(SUMMARY_CSV, index=False)
print(f"\nAppended to {SUMMARY_CSV} -- run this same script for each remaining district.")

try:
    from google.colab import files
    files.download(SUMMARY_CSV)
    files.download(RESULTS_JSONL)
except Exception:
    pass


RUNNING DISTRICT: Madhubani
Loaded 1102 villages from /content/village_codes_Madhubani.csv

--- Fetch pass 1/3: 249 villages remaining ---
  [50/249] fetched this pass
  [100/249] fetched this pass
  [150/249] fetched this pass
  [200/249] fetched this pass
Pass 1 done: 247 succeeded, 2 failed.

--- Fetch pass 2/3: 2 villages remaining ---
Pass 2 done: 0 succeeded, 2 failed.

--- Fetch pass 3/3: 2 villages remaining ---
Pass 3 done: 0 succeeded, 2 failed.

FETCH SUMMARY: 1100/1102 fetched (2 missing)

--- Computing field boundary count + area ---

District: Madhubani
Villages processed: 1099/1102
Field Boundaries Identified (total plots): 3,640,729
Area Delineated (ha): 353,221.40

Appended to district_area_summary.csv -- run this same script for each remaining district.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
LEAN target-metrics pipeline -- Field Boundaries + Area Delineated (ha) only
--------------------------------------------------------------------------------
Unlike the full pipeline, this does NOT write per-village shapefiles. It only
needs two numbers per district:
    - Field Boundaries Identified  = total plot count
    - Area Delineated (ha)         = total area, in HECTARES

Much lighter/faster since nothing is written to disk except the raw API
responses (for safety) and the final summary -- no shapefile writing, no
per-village GeoDataFrame overhead.

USAGE -- edit these two lines, run:
    DISTRICT_NAME = "Gaya"
    VILLAGE_CSV = "village_codes_Gaya.csv"

Appends its result to district_area_summary.csv each time you run it for a
new district, so after running all 4 you have one combined comparison table.
"""

DISTRICT_NAME = "Samastipur"
VILLAGE_CSV = "/content/village_codes_Samastipur.csv"


# ============================================================
# INSTALL + IMPORTS
# ============================================================
import subprocess
subprocess.run(["pip", "install", "-q", "requests", "shapely", "pyproj", "pandas"], check=True)

import requests, json, csv, os, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely import wkt as shapely_wkt
from shapely.geometry import shape
import pandas as pd

RESULTS_JSONL = f"results_{DISTRICT_NAME.lower()}.jsonl"
STATE_CODE = "10"
MAX_WORKERS = 5
TIMEOUT_SECONDS = 60
MAX_RETRY_PASSES = 3
SQM_PER_HECTARE = 10000.0
SQM_PER_ACRE = 4046.8564224
SUMMARY_CSV = "district_area_summary.csv"

print(f"{'='*70}\nRUNNING DISTRICT: {DISTRICT_NAME}\n{'='*70}")


# ============================================================
# TOKEN-MANAGED API CLIENT
# ============================================================
BASE_URL = "https://bhunaksha.bihar.gov.in/rest/GetMapInstanceMap"
GENERATE_TOKEN_URL = f"{BASE_URL}/generateToken"
GET_GEOM_URL = f"{BASE_URL}/getGeom"

class BhunakshaClient:
    def __init__(self):
        self.token, self.token_expiry = None, None
        self._lock = threading.Lock()

    def _parse_expiry(self, s, at):
        try:
            v, unit = s.strip().split()
            v = int(v)
            unit = unit.lower()
            delta = timedelta(minutes=v) if "minute" in unit else timedelta(seconds=v) if "second" in unit else timedelta(hours=v) if "hour" in unit else timedelta(minutes=30)
        except Exception:
            delta = timedelta(minutes=30)
        return at + delta - timedelta(seconds=60)

    def _generate_token_locked(self):
        at = datetime.now()
        r = requests.post(GENERATE_TOKEN_URL, timeout=15)
        r.raise_for_status()
        data = r.json()
        if not data.get("success"):
            raise RuntimeError(f"Token generation failed: {data}")
        self.token = data["token"]
        self.token_expiry = self._parse_expiry(data.get("expiresIn", "30 minutes"), at)

    def _get_token(self):
        with self._lock:
            if self.token is None or datetime.now() >= self.token_expiry:
                self._generate_token_locked()
            return self.token

    def get_geom(self, state, code, map_type, sheet_no):
        token = self._get_token()
        payload = {"state": state, "lgdVillageCode": code, "mapType": map_type, "sheetNo": sheet_no, "token": token}
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        if r.status_code in (401, 403):
            with self._lock:
                self._generate_token_locked()
                token = self.token
            payload["token"] = token
            r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        r.raise_for_status()
        return r.json()


# ============================================================
# STEP 1: FETCH (raw responses saved for safety, multi-pass retry)
# ============================================================
def load_village_rows(csv_path):
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        raw_rows = list(csv.DictReader(f))
    def norm(k): return (k or "").strip().lower().replace(" ", "_")
    headers = list(raw_rows[0].keys())
    nmap = {norm(k): k for k in headers}
    if "lgd_village_code" not in nmap:
        raise KeyError(f"No lgd_village_code column found. Actual columns: {headers}")
    rows = []
    for row in raw_rows:
        rows.append({
            "lgd_village_code": str(row[nmap["lgd_village_code"]]).strip(),
            "map_type": row.get(nmap.get("map_type", ""), "CS").strip() if nmap.get("map_type") else "CS",
            "sheet_no": row.get(nmap.get("sheet_no", ""), "00").strip() if nmap.get("sheet_no") else "00",
        })
    return rows

def get_fetched_codes():
    done = set()
    if os.path.exists(RESULTS_JSONL):
        with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["lgd_village_code"])
                except Exception:
                    continue
    return done

village_rows = load_village_rows(VILLAGE_CSV)
print(f"Loaded {len(village_rows)} villages from {VILLAGE_CSV}")

client = BhunakshaClient()
write_lock = threading.Lock()

def fetch_one(row):
    return row["lgd_village_code"], row["map_type"], row["sheet_no"], client.get_geom(STATE_CODE, row["lgd_village_code"], row["map_type"], row["sheet_no"])

for attempt in range(1, MAX_RETRY_PASSES + 1):
    already_done = get_fetched_codes()
    todo = [r for r in village_rows if r["lgd_village_code"] not in already_done]
    if not todo:
        print("All villages fetched.")
        break

    print(f"\n--- Fetch pass {attempt}/{MAX_RETRY_PASSES}: {len(todo)} villages remaining ---")
    done_count, fail_count = 0, 0

    with open(RESULTS_JSONL, "a", encoding="utf-8") as out_f:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_one, row): row for row in todo}
            for future in as_completed(futures):
                row = futures[future]
                try:
                    code, mt, sn, geom = future.result()
                    with write_lock:
                        out_f.write(json.dumps({"lgd_village_code": code, "map_type": mt, "sheet_no": sn, "geometry": geom}) + "\n")
                        out_f.flush()
                        done_count += 1
                        if done_count % 50 == 0:
                            print(f"  [{done_count}/{len(todo)}] fetched this pass")
                except Exception:
                    fail_count += 1

    print(f"Pass {attempt} done: {done_count} succeeded, {fail_count} failed.")

final_fetched = get_fetched_codes()
final_missing = len(village_rows) - len(final_fetched)
print(f"\nFETCH SUMMARY: {len(final_fetched)}/{len(village_rows)} fetched ({final_missing} missing)")


# ============================================================
# STEP 2: COMPUTE TOTALS ONLY -- no shapefile writing
# ============================================================
print(f"\n--- Computing field boundary count + area ---")

total_plots = 0
total_area_sqm = 0.0
villages_processed = 0

with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        features = (record.get("geometry") or {}).get("features", [])
        if not features:
            continue
        villages_processed += 1

        village_areas = []
        for feat in features:
            props = feat.get("properties", {})
            geom = None
            wkt_str = props.get("the_geom")
            if wkt_str:
                try: geom = shapely_wkt.loads(wkt_str)
                except Exception: geom = None
            if geom is None:
                raw_geom = feat.get("geometry")
                if raw_geom:
                    try: geom = shape(raw_geom)
                    except Exception: geom = None
            if geom is None:
                continue

            total_plots += 1
            raw_area = props.get("area_acres") if props.get("area_acres") is not None else props.get("area")
            if raw_area is not None:
                try:
                    village_areas.append(float(raw_area))
                except (ValueError, TypeError):
                    pass

        if village_areas:
            median_val = sorted(village_areas)[len(village_areas)//2]
            # AUTO UNIT DETECTION: same heuristic as before -- values in the
            # thousands+ are sq meters, smaller values are already acres
            if median_val > 50:
                village_sum_sqm = sum(village_areas)  # already sqm
            else:
                village_sum_sqm = sum(village_areas) * SQM_PER_ACRE  # acres -> sqm
            total_area_sqm += village_sum_sqm

total_area_ha = total_area_sqm / SQM_PER_HECTARE

print(f"\n{'='*60}")
print(f"District: {DISTRICT_NAME}")
print(f"Villages processed: {villages_processed}/{len(village_rows)}")
print(f"Field Boundaries Identified (total plots): {total_plots:,}")
print(f"Area Delineated (ha): {total_area_ha:,.2f}")
print(f"{'='*60}")

# append to running summary CSV
row = {
    "District": DISTRICT_NAME,
    "Field Boundaries Identified": total_plots,
    "Area Delineated (ha)": round(total_area_ha, 2),
    "Villages Processed": villages_processed,
    "Villages Total": len(village_rows),
}
if os.path.exists(SUMMARY_CSV):
    df = pd.read_csv(SUMMARY_CSV)
    df = df[df["District"] != DISTRICT_NAME]  # replace if re-running same district
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])
df.to_csv(SUMMARY_CSV, index=False)
print(f"\nAppended to {SUMMARY_CSV} -- run this same script for each remaining district.")

try:
    from google.colab import files
    files.download(SUMMARY_CSV)
    files.download(RESULTS_JSONL)
except Exception:
    pass

RUNNING DISTRICT: Samastipur
Loaded 1837 villages from /content/village_codes_Samastipur.csv

--- Fetch pass 1/3: 1837 villages remaining ---
  [50/1837] fetched this pass
  [100/1837] fetched this pass
  [150/1837] fetched this pass
  [200/1837] fetched this pass
  [250/1837] fetched this pass
  [300/1837] fetched this pass
  [350/1837] fetched this pass
  [400/1837] fetched this pass
  [450/1837] fetched this pass
  [500/1837] fetched this pass
  [550/1837] fetched this pass
  [600/1837] fetched this pass
  [650/1837] fetched this pass
  [700/1837] fetched this pass
  [750/1837] fetched this pass
  [800/1837] fetched this pass
  [850/1837] fetched this pass
  [900/1837] fetched this pass
  [950/1837] fetched this pass
  [1000/1837] fetched this pass
  [1050/1837] fetched this pass
  [1100/1837] fetched this pass
  [1150/1837] fetched this pass
  [1200/1837] fetched this pass
  [1250/1837] fetched this pass
  [1300/1837] fetched this pass
  [1350/1837] fetched this pass
  [1400/1837] 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
LEAN target-metrics pipeline -- Field Boundaries + Area Delineated (ha) only
--------------------------------------------------------------------------------
Unlike the full pipeline, this does NOT write per-village shapefiles. It only
needs two numbers per district:
    - Field Boundaries Identified  = total plot count
    - Area Delineated (ha)         = total area, in HECTARES

Much lighter/faster since nothing is written to disk except the raw API
responses (for safety) and the final summary -- no shapefile writing, no
per-village GeoDataFrame overhead.

USAGE -- edit these two lines, run:
    DISTRICT_NAME = "Gaya"
    VILLAGE_CSV = "village_codes_Gaya.csv"

Appends its result to district_area_summary.csv each time you run it for a
new district, so after running all 4 you have one combined comparison table.
"""

DISTRICT_NAME = "Saran"
VILLAGE_CSV = "/content/village_codes_Saran.csv"


# ============================================================
# INSTALL + IMPORTS
# ============================================================
import subprocess
subprocess.run(["pip", "install", "-q", "requests", "shapely", "pyproj", "pandas"], check=True)

import requests, json, csv, os, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely import wkt as shapely_wkt
from shapely.geometry import shape
import pandas as pd

RESULTS_JSONL = f"results_{DISTRICT_NAME.lower()}.jsonl"
STATE_CODE = "10"
MAX_WORKERS = 5
TIMEOUT_SECONDS = 60
MAX_RETRY_PASSES = 3
SQM_PER_HECTARE = 10000.0
SQM_PER_ACRE = 4046.8564224
SUMMARY_CSV = "district_area_summary.csv"

print(f"{'='*70}\nRUNNING DISTRICT: {DISTRICT_NAME}\n{'='*70}")


# ============================================================
# TOKEN-MANAGED API CLIENT
# ============================================================
BASE_URL = "https://bhunaksha.bihar.gov.in/rest/GetMapInstanceMap"
GENERATE_TOKEN_URL = f"{BASE_URL}/generateToken"
GET_GEOM_URL = f"{BASE_URL}/getGeom"

class BhunakshaClient:
    def __init__(self):
        self.token, self.token_expiry = None, None
        self._lock = threading.Lock()

    def _parse_expiry(self, s, at):
        try:
            v, unit = s.strip().split()
            v = int(v)
            unit = unit.lower()
            delta = timedelta(minutes=v) if "minute" in unit else timedelta(seconds=v) if "second" in unit else timedelta(hours=v) if "hour" in unit else timedelta(minutes=30)
        except Exception:
            delta = timedelta(minutes=30)
        return at + delta - timedelta(seconds=60)

    def _generate_token_locked(self):
        at = datetime.now()
        r = requests.post(GENERATE_TOKEN_URL, timeout=15)
        r.raise_for_status()
        data = r.json()
        if not data.get("success"):
            raise RuntimeError(f"Token generation failed: {data}")
        self.token = data["token"]
        self.token_expiry = self._parse_expiry(data.get("expiresIn", "30 minutes"), at)

    def _get_token(self):
        with self._lock:
            if self.token is None or datetime.now() >= self.token_expiry:
                self._generate_token_locked()
            return self.token

    def get_geom(self, state, code, map_type, sheet_no):
        token = self._get_token()
        payload = {"state": state, "lgdVillageCode": code, "mapType": map_type, "sheetNo": sheet_no, "token": token}
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        if r.status_code in (401, 403):
            with self._lock:
                self._generate_token_locked()
                token = self.token
            payload["token"] = token
            r = requests.post(GET_GEOM_URL, data=payload, headers=headers, timeout=TIMEOUT_SECONDS)
        r.raise_for_status()
        return r.json()


# ============================================================
# STEP 1: FETCH (raw responses saved for safety, multi-pass retry)
# ============================================================
def load_village_rows(csv_path):
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        raw_rows = list(csv.DictReader(f))
    def norm(k): return (k or "").strip().lower().replace(" ", "_")
    headers = list(raw_rows[0].keys())
    nmap = {norm(k): k for k in headers}
    if "lgd_village_code" not in nmap:
        raise KeyError(f"No lgd_village_code column found. Actual columns: {headers}")
    rows = []
    for row in raw_rows:
        rows.append({
            "lgd_village_code": str(row[nmap["lgd_village_code"]]).strip(),
            "map_type": row.get(nmap.get("map_type", ""), "CS").strip() if nmap.get("map_type") else "CS",
            "sheet_no": row.get(nmap.get("sheet_no", ""), "00").strip() if nmap.get("sheet_no") else "00",
        })
    return rows

def get_fetched_codes():
    done = set()
    if os.path.exists(RESULTS_JSONL):
        with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["lgd_village_code"])
                except Exception:
                    continue
    return done

village_rows = load_village_rows(VILLAGE_CSV)
print(f"Loaded {len(village_rows)} villages from {VILLAGE_CSV}")

client = BhunakshaClient()
write_lock = threading.Lock()

def fetch_one(row):
    return row["lgd_village_code"], row["map_type"], row["sheet_no"], client.get_geom(STATE_CODE, row["lgd_village_code"], row["map_type"], row["sheet_no"])

for attempt in range(1, MAX_RETRY_PASSES + 1):
    already_done = get_fetched_codes()
    todo = [r for r in village_rows if r["lgd_village_code"] not in already_done]
    if not todo:
        print("All villages fetched.")
        break

    print(f"\n--- Fetch pass {attempt}/{MAX_RETRY_PASSES}: {len(todo)} villages remaining ---")
    done_count, fail_count = 0, 0

    with open(RESULTS_JSONL, "a", encoding="utf-8") as out_f:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_one, row): row for row in todo}
            for future in as_completed(futures):
                row = futures[future]
                try:
                    code, mt, sn, geom = future.result()
                    with write_lock:
                        out_f.write(json.dumps({"lgd_village_code": code, "map_type": mt, "sheet_no": sn, "geometry": geom}) + "\n")
                        out_f.flush()
                        done_count += 1
                        if done_count % 50 == 0:
                            print(f"  [{done_count}/{len(todo)}] fetched this pass")
                except Exception:
                    fail_count += 1

    print(f"Pass {attempt} done: {done_count} succeeded, {fail_count} failed.")

final_fetched = get_fetched_codes()
final_missing = len(village_rows) - len(final_fetched)
print(f"\nFETCH SUMMARY: {len(final_fetched)}/{len(village_rows)} fetched ({final_missing} missing)")


# ============================================================
# STEP 2: COMPUTE TOTALS ONLY -- no shapefile writing
# ============================================================
print(f"\n--- Computing field boundary count + area ---")

total_plots = 0
total_area_sqm = 0.0
villages_processed = 0

with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        features = (record.get("geometry") or {}).get("features", [])
        if not features:
            continue
        villages_processed += 1

        village_areas = []
        for feat in features:
            props = feat.get("properties", {})
            geom = None
            wkt_str = props.get("the_geom")
            if wkt_str:
                try: geom = shapely_wkt.loads(wkt_str)
                except Exception: geom = None
            if geom is None:
                raw_geom = feat.get("geometry")
                if raw_geom:
                    try: geom = shape(raw_geom)
                    except Exception: geom = None
            if geom is None:
                continue

            total_plots += 1
            raw_area = props.get("area_acres") if props.get("area_acres") is not None else props.get("area")
            if raw_area is not None:
                try:
                    village_areas.append(float(raw_area))
                except (ValueError, TypeError):
                    pass

        if village_areas:
            median_val = sorted(village_areas)[len(village_areas)//2]
            # AUTO UNIT DETECTION: same heuristic as before -- values in the
            # thousands+ are sq meters, smaller values are already acres
            if median_val > 50:
                village_sum_sqm = sum(village_areas)  # already sqm
            else:
                village_sum_sqm = sum(village_areas) * SQM_PER_ACRE  # acres -> sqm
            total_area_sqm += village_sum_sqm

total_area_ha = total_area_sqm / SQM_PER_HECTARE

print(f"\n{'='*60}")
print(f"District: {DISTRICT_NAME}")
print(f"Villages processed: {villages_processed}/{len(village_rows)}")
print(f"Field Boundaries Identified (total plots): {total_plots:,}")
print(f"Area Delineated (ha): {total_area_ha:,.2f}")
print(f"{'='*60}")

# append to running summary CSV
row = {
    "District": DISTRICT_NAME,
    "Field Boundaries Identified": total_plots,
    "Area Delineated (ha)": round(total_area_ha, 2),
    "Villages Processed": villages_processed,
    "Villages Total": len(village_rows),
}
if os.path.exists(SUMMARY_CSV):
    df = pd.read_csv(SUMMARY_CSV)
    df = df[df["District"] != DISTRICT_NAME]  # replace if re-running same district
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])
df.to_csv(SUMMARY_CSV, index=False)
print(f"\nAppended to {SUMMARY_CSV} -- run this same script for each remaining district.")

try:
    from google.colab import files
    files.download(SUMMARY_CSV)
    files.download(RESULTS_JSONL)
except Exception:
    pass

RUNNING DISTRICT: Saran
Loaded 3378 villages from /content/village_codes_Saran.csv

--- Fetch pass 1/3: 10 villages remaining ---
Pass 1 done: 10 succeeded, 0 failed.
All villages fetched.

FETCH SUMMARY: 1789/3378 fetched (1589 missing)

--- Computing field boundary count + area ---

District: Saran
Villages processed: 2582/3378
Field Boundaries Identified (total plots): 4,397,762
Area Delineated (ha): 1,335,233.50

Appended to district_area_summary.csv -- run this same script for each remaining district.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>